In [ ]:
# import duckdb
# import glob
# import os

# con = duckdb.connect()
# con.execute("INSTALL sqlite; LOAD sqlite;")
# con.execute("ATTACH 'medical_data.db' AS sqlite_db (TYPE SQLITE);")

# for csv_file in glob.glob("../synthea/output/csv/*.csv"):
#     table_name = os.path.splitext(os.path.basename(csv_file))[0]
#     print(f"Importing {table_name}...")
#     con.execute(f"CREATE TABLE sqlite_db.{table_name} AS SELECT * FROM read_csv_auto('{csv_file}');")

# print("Done!")

Importing allergies...
Importing careplans...
Importing claims...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Importing claims_transactions...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Importing conditions...


...

Importing procedures...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Importing providers...
Importing supplies...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done!


In [14]:
import pandas as pd
import numpy as np
from SQL_query import SQLQuery
SQL = SQLQuery("medical_data.db") 
query = """
WITH LATEST_PREGNANCY AS (
    SELECT
        PATIENT,
        MAX(DATE(SUBSTR(START, 1, 10))) AS PREGNANCY_START,
        DATE(MAX(SUBSTR(START, 1, 10)), '+270 days') AS EST_DELIVERY_DATE
    FROM encounters
    WHERE DESCRIPTION = 'Prenatal initial visit (regime/therapy)'
    GROUP BY PATIENT
),
LATEST_PREG_DIA AS (
    SELECT
        o.PATIENT,
        MAX(DATE(SUBSTR(o.DATE, 1, 10))) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Diastolic Blood Pressure' THEN o.VALUE END) AS Diastolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Diastolic Blood Pressure' AND DATE(SUBSTR(o.DATE, 1, 10)) <= lp.EST_DELIVERY_DATE
    GROUP BY o.PATIENT
),
LATEST_PREG_SYS AS (
    SELECT
        o.PATIENT,
        MAX(DATE(SUBSTR(o.DATE, 1, 10))) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Systolic Blood Pressure' THEN o.VALUE END) AS Systolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Systolic Blood Pressure' AND DATE(SUBSTR(o.DATE, 1, 10)) <= lp.EST_DELIVERY_DATE
    GROUP BY o.PATIENT
),
DEATH_CAUSE AS (
    SELECT
        d.PATIENT,
        d.VALUE AS CAUSE_OF_DEATH
    FROM observations as d
    WHERE d.DESCRIPTION = 'Cause of Death [US Standard Certificate of Death]'
)
SELECT 
    t1.Id,
    DATE(t1.BIRTHDATE) AS BIRTHDATE,
    DATE(t1.DEATHDATE) AS DEATHDATE,
    t1.RACE,
    t1.ETHNICITY,
    t1.GENDER,
    t1.COUNTY,
    t1.HEALTHCARE_EXPENSES,
    t1.HEALTHCARE_COVERAGE,
    t1.INCOME,
    t1.HEALTHCARE_EXPENSES - t1.INCOME AS RELATIVE_EXPENSES,
    DATE(t2.PREGNANCY_START) AS PREGNANCY_START,
    DATE(t2.EST_DELIVERY_DATE) AS EST_DELIVERY_DATE,
    DATE(t3.OBSERVATION_DATE) AS BP_OBSERVATION_DATE,
    t3.Diastolic_Blood_Pressure,
    t4.Systolic_Blood_Pressure,
    t8.CAUSE_OF_DEATH
    
FROM patients as t1
INNER JOIN LATEST_PREGNANCY as t2 ON t1.Id = t2.PATIENT
LEFT JOIN LATEST_PREG_DIA as t3 ON t1.Id = t3.PATIENT
LEFT JOIN LATEST_PREG_SYS as t4 ON t1.Id = t4.PATIENT
LEFT JOIN DEATH_CAUSE as t8 ON t1.Id = t8.PATIENT
"""
LAfemMed = SQL.execute_query(query)
LAfemMed['BIRTHDATE'] = pd.to_datetime(LAfemMed['BIRTHDATE'])
LAfemMed['DEATHDATE'] = pd.to_datetime(LAfemMed['DEATHDATE'])
LAfemMed['PREGNANCY_START'] = pd.to_datetime(LAfemMed['PREGNANCY_START'])
LAfemMed['EST_DELIVERY_DATE'] = pd.to_datetime(LAfemMed['EST_DELIVERY_DATE'])
LAfemMed['BP_OBSERVATION_DATE'] = pd.to_datetime(LAfemMed['BP_OBSERVATION_DATE'])
LAfemMed['Days_Pregancy_to_Death'] = (LAfemMed['DEATHDATE'] - LAfemMed['PREGNANCY_START']).dt.days
LAfemMed['Maternal_Mortality'] = np.where(LAfemMed['Days_Pregancy_to_Death'] <= 270+42, 1, 0)

Maternal_Deaths = LAfemMed[LAfemMed['Maternal_Mortality'] == 1]

# Preprocess Data

In [15]:
from sklearn.preprocessing import StandardScaler
numeric_cols = ['INCOME', 'RELATIVE_EXPENSES', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure']
all_cols = ['INCOME', 'RELATIVE_EXPENSES', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure', 'Maternal_Mortality']
final_data = LAfemMed[all_cols].dropna()
scale = StandardScaler()
X = scale.fit_transform(final_data[numeric_cols])
y = final_data['Maternal_Mortality'].to_numpy().astype(float)

In [16]:
from sklearn.naive_bayes import GaussianNB
GNB = GaussianNB()
GNB.fit(X,y)
pred = GNB.predict(X)

In [17]:
from LogisticRegression import LR_class
LR = LR_class()
LR_fit = LR.LogRes(X,y)
LR_fit.predict(X)

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1861: UserWarning: l1_ratios parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(


array([0., 0., 0., ..., 0., 0., 0.])

In [18]:
from SVM import SVM_class
svm = SVM_class()
svm.best_fit(X,y)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan]
  warnings.warn(


{'best': SVC(C=0.6, gamma=0.01, random_state=42),
 'model': RandomizedSearchCV(cv=5, estimator=SVC(random_state=42), n_iter=20, n_jobs=8,
                    param_distributions={'C': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
                                         'gamma': [0.01, 0.05, 0.1, 0.5]},
                    scoring='neg_log_loss', verbose=2)}

In [ ]:
import RandForest
rf = RandForest.RF_hyperparams()
rf.best_fit(X,y)

In [ ]:
from NNmod import NN_mod
in_size = X.shape[1]
HidLay_combos = [[], [], []]
nn_mort = NN_mod(epochs=1000)
nn_mort.fit(X, y)

In [21]:
from XGBoost import XGBclass
xgb = XGBclass(X, y)
best_params = xgb.xgb_Kfold(k=5)
best_params

  0%|          | 0/500 [00:00<?, ?trial/s, best loss=?]

C:\Users\ej_la\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:226: UserWarning: [22:27:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "n_estimators" } are not used.

  return getattr(self.bst, name)(*args, **kwargs)

C:\Users\ej_la\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:232: UserWarning: [22:27:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "n_estimators" } are not used.

  self.bst.update(self.dtrain, iteration, fobj)



  0%|          | 1/500 [00:01<08:45,  1.05s/trial, best loss: 0.05790922097944362]

C:\Users\ej_la\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:226: UserWarning: [22:27:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "n_estimators" } are not used.

  return getattr(self.bst, name)(*args, **kwargs)

C:\Users\ej_la\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:232: UserWarning: [22:27:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "n_estimators" } are not used.

  self.bst.update(self.dtrain, iteration, fobj)



  0%|          | 2/500 [00:01<05:46,  1.44trial/s, best loss: 0.05790922097944362]

...

C:\Users\ej_la\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:226: UserWarning: [22:28:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "n_estimators" } are not used.

  return getattr(self.bst, name)(*args, **kwargs)

C:\Users\ej_la\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:232: UserWarning: [22:28:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "n_estimators" } are not used.

  self.bst.update(self.dtrain, iteration, fobj)



  6%|▋         | 32/500 [00:59<19:58,  2.56s/trial, best loss: 0.05540913184675126]

C:\Users\ej_la\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:226: UserWarning: [22:28:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "n_estimators" } are not used.

  return getattr(self.bst, name)(*args, **kwargs)

C:\Users\ej_la\AppData\Roaming\Python\Python312\site-packages\xgboost\training.py:232: UserWarning: [22:28:23] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "n_estimators" } are not used.

  self.bst.update(self.dtrain, iteration, fobj)



  7%|▋         | 33/500 [01:01<14:27,  1.86s/trial, best loss: 0.05540913184675126]


{'colsample_bylevel': 0.9927995232911688,
 'colsample_bynode': 0.5782029834046934,
 'colsample_bytree': 0.8489239032888045,
 'eta': 0.20947804096598321,
 'eval_metric': 'aucpr',
 'gamma': 5.365976191463984,
 'max_delta_step': 2,
 'max_depth': 4,
 'min_child_weight': 1,
 'n_estimators': 170,
 'objective': 'binary:logistic',
 'reg_alpha': 7.327767616500402,
 'reg_lambda': 0.2828521884591474,
 'seed': 44,
 'subsample': 0.9085333858537321}